# 청킹 결과 평가 (Chunking Evaluation)

이 노트북은 **청킹을 설계하는 곳이 아니라 채점하는 곳**이다. `apps/ai/app/rag/chunking/`이
만든 청크를 받아 **검색에 쓸 수 있는 상태인지**를 판정한다.
전략과 규칙의 근거는 `llm_wiki/_context/chunking-strategy.md`에 있다.

입력은 `docling_parsing_test.ipynb`가 떨군 파싱 덤프(`output/layout/layout_result.csv`,
11종 4,388요소)다. **docling을 재실행하지 않는다** — 덤프에 교정(C1~C7)과 청킹만 다시 걸므로
수 초 안에 끝나고, 별도 conda 환경도 필요 없다.

| # | 평가 영역 | 핵심 질문 | 가중치 |
|---|---|---|---|
| 1 | **Structure Integrity** | 원문이 유실·중복·혼입 없이 청크로 옮겨졌는가? | 0.40 |
| 2 | **Context Sufficiency** | 청크 하나만 보고 어느 조문인지 알 수 있는가? | 0.30 |
| 3 | **Size Fitness** | 임베딩·검색에 맞는 크기인가? | 0.30 |
| — | **Retrieval Effectiveness** | Recall@K·MRR | **N/A** |

> ⚠️ **Retrieval Effectiveness는 점수화하지 않는다.** 임베딩 모델이 미확정이고 질의 정답셋이
> 없다(전략 문서 §11 결정대기 ①·②). **정답 없는 항목은 점수화 금지** — 파싱 평가 노트북과
> 같은 규칙이며, 해당 가중치는 나머지 영역에 재분배한다.

**이 노트북이 자동으로 판정할 수 있는 것과 없는 것**

| 프롬프트 요구 항목 | 자동 판정 | 방법 |
|---|---|---|
| ① Context Completeness | ⚠️ 대리지표 | 헤더·인용 보유율 + 지시어로 시작하는 청크 비율. **최종 판단은 사람 검수(§10)** |
| ② Semantic Integrity | ✅ | 요소 커버리지·소유 중복·조 경계 순수성 |
| ③ Boundary Quality | ✅ | 강제 분할 수 · 요소 경계 준수 |
| ④ Hierarchy Preservation | ✅ | 조/장 라벨 보유율 |
| ⑤ Table Integrity | ✅ | 표 원자성 · 행 보존율 · 헤더 반복 |
| ⑥ Retrieval Suitability | ⚠️ 크기만 | 분포는 재고, 검색 성능은 **N/A** |


---
## 1. 실행 환경 · 청크 생성


In [1]:
from __future__ import annotations

import json
import random
import re
import statistics as st
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 220)


def _resolve_base_dir() -> Path:
    """CWD가 레포 루트든 docling_eval 내부든 같은 docling_eval 을 가리키게 한다."""
    cwd = Path.cwd().resolve()
    for cand in (cwd, *cwd.parents):
        if cand.name == "docling_eval":
            return cand
        if (cand / "docling_eval").is_dir() or (cand / "tiger_inc").is_dir():
            return cand / "docling_eval"
    return cwd / "docling_eval"


BASE_DIR = _resolve_base_dir()
REPO_ROOT = BASE_DIR.parent
AI_APP = REPO_ROOT / "apps" / "ai"
if str(AI_APP) not in sys.path:
    sys.path.insert(0, str(AI_APP))          # 청킹 구현은 앱 코드가 정본이다

# ── 입력 ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = BASE_DIR / "output"
LAYOUT_CSV = OUTPUT_DIR / "layout" / "layout_result.csv"     # docling 요소 전수 덤프
TABLES_DIR = OUTPUT_DIR / "tables"                           # docling 표 격자

# ── 출력 ────────────────────────────────────────────────────────────────────
CHUNK_DIR = OUTPUT_DIR / "chunking"
CHUNK_DIR.mkdir(parents=True, exist_ok=True)

# ▼▼▼ 평가 대상·파라미터 — 바꿀 곳은 여기뿐 ▼▼▼
TARGET_DOCS = None            # None = 전체. 일부만: ["법인카드_사용규정"]
BUDGET_OVERRIDE = None        # None = 기본 예산. 예: dict(target=500, max=800)
SWEEP_MAXES = [600, 800, 1200, 1600, 2400]   # §9 예산 민감도
REVIEW_SAMPLE_N = 40          # §10 사람 검수 시트 표본 수
RANDOM_SEED = 29
# ▲▲▲

if not LAYOUT_CSV.exists():
    raise FileNotFoundError(
        f"파싱 덤프가 없습니다: {LAYOUT_CSV}\n"
        "→ 먼저 docling_eval/docling_parsing_test.ipynb 를 끝까지 실행하세요."
    )

from app.rag.chunking import chunk_document                  # noqa: E402
from app.rag.chunking.model import Budget                    # noqa: E402
from app.rag.parsing import dump as dump_mod                 # noqa: E402
from app.rag.parsing.corrections import pipeline             # noqa: E402

BUDGET = Budget(**BUDGET_OVERRIDE) if BUDGET_OVERRIDE else Budget()
random.seed(RANDOM_SEED)

print(f"python      : {sys.version.split()[0]}")
print(f"pandas      : {pd.__version__}")
print(f"BASE_DIR    : {BASE_DIR}")
print(f"청킹 구현   : {AI_APP / 'app' / 'rag' / 'chunking'}")
print(f"평가 산출물 : {CHUNK_DIR}")
print(f"예산        : target={BUDGET.target} max={BUDGET.max} hard={BUDGET.hard} "
      f"min_merge={BUDGET.min_merge} table_max_rows={BUDGET.table_max_rows}")


python      : 3.12.13
pandas      : 3.0.5
BASE_DIR    : D:\project\SKN29-FINAL-1TEAM\docling_eval
청킹 구현   : D:\project\SKN29-FINAL-1TEAM\apps\ai\app\rag\chunking
평가 산출물 : D:\project\SKN29-FINAL-1TEAM\docling_eval\output\chunking
예산        : target=800 max=1200 hard=2000 min_merge=250 table_max_rows=40


### 1-1. 파싱 덤프 → 교정(C1~C7) → 청킹

운영 경로(`engine.convert()`)와 검증 경로(`dump.load_all()`)는 **교정·청킹 계층을 공유**한다.
여기서 재는 것이 곧 운영에서 나오는 것이다.


In [2]:
DOCS: dict = {}          # {문서명: ParsedDoc}
CHUNKS: dict = {}        # {문서명: list[Chunk]}
REPORTS: dict = {}       # {문서명: ChunkReport}

for name, doc in dump_mod.load_all(LAYOUT_CSV, TABLES_DIR).items():
    if TARGET_DOCS and name not in TARGET_DOCS:
        continue
    pipeline.run(doc)                       # 교정 — 청킹은 교정된 계층에만 의존한다
    chunks, report = chunk_document(doc, BUDGET)
    DOCS[name], CHUNKS[name], REPORTS[name] = doc, chunks, report

ALL_CHUNKS = [c for cs in CHUNKS.values() for c in cs]
LEAVES = [c for c in ALL_CHUNKS if c.chunk_role != "parent"]

print(f"문서 {len(DOCS)}종 · 요소 {sum(len(d.body()) for d in DOCS.values()):,}개")
print(f"청크 {len(ALL_CHUNKS):,}개 (잎 {len(LEAVES):,} / 부모 {len(ALL_CHUNKS) - len(LEAVES)})")
print()
for name, doc in DOCS.items():
    rep = REPORTS[name]
    warn = f"  ⚠ {'; '.join(rep.warnings)}" if rep.warnings else ""
    print(f"  {name:<20} [{doc.profile:<10}] 청크 {len(CHUNKS[name]):>4}  분할 {dict(rep.splits)}{warn}")


문서 11종 · 요소 3,717개
청크 888개 (잎 799 / 부모 89)

  법인카드_사용규정            [REGULATION] 청크   25  분할 {}
  부서소개                 [DIAGRAM   ] 청크    6  분할 {}
  업무추진비_사용규정           [REGULATION] 청크   23  분할 {'clause': 1}
  조직도                  [DIAGRAM   ] 청크   10  분할 {}
  조직설계_상세기획서           [DIAGRAM   ] 청크   30  분할 {}
  직급체계                 [DIAGRAM   ] 청크    9  분할 {}
  출장비_사용규정             [REGULATION] 청크   24  분할 {}
  회식_운영규정              [REGULATION] 청크   31  분할 {'suspect_article_ref': 1}
  법인세법                 [LAW       ] 청크  425  분할 {'front_matter': 2, 'suspect_article_heading': 1, 'suspect_article_ref': 1, 'polluted_heading': 42, 'article_start_in_body': 85, 'clause': 59, 'list_item': 83}
  부가가치세법               [LAW       ] 청크  130  분할 {'front_matter': 2, 'polluted_heading': 15, 'article_start_in_body': 60, 'clause': 19}
  여신전문금융업법             [LAW       ] 청크  175  분할 {'front_matter': 2, 'polluted_heading': 7, 'article_start_in_body': 26, 'clause': 6, 'list_item': 45}


### 1-2. 문서 메타(C7) 승격 점검 — 시행일이 실제로 잡혔는가

`ParsedDoc.meta`는 **표지에서 승격된 문서 단위 값**이고, 청킹이 이를 전 청크에 복제해
`doc_effective_date` 등으로 Chroma까지 내보낸다. 그런데 이 노트북은 지금까지 청크만 보고
**문서 메타는 한 번도 화면에 내놓지 않았다** — 그래서 결손이 오래 안 보였다.

📏 실제 사고: docling이 같은 표지 레이아웃을 법인카드·출장비에선 `table`(grid)로,
업무추진비·회식에선 개별 `paragraph`로 내놓는다. `grid`만 보던 C7은 후자 2종에서
`effective_date`를 통째로 놓쳤고, **규정 4종이 모두 같은 시행일(2026.8.1)이라는 우연**이
그 결손을 가려 줬다. 아래 표에서 `표지 형태` 열이 원인을, `⚠` 열이 결과를 보여 준다.

> 값이 비면 **경고를 띄운다.** 조용히 `None`이 지나가는 것이 이 결손의 본질이었다.

In [ ]:
# REGULATION·LAW는 시행일이 있어야 한다. DIAGRAM은 C7 자체가 skip이라 대상 아님.
_META_REQUIRED = {"REGULATION": ("effective_date",), "LAW": ("effective_date", "doc_no")}

meta_rows, meta_missing = [], []
for name, doc in DOCS.items():
    # 표지 표는 REGULATION만의 구조다. 법령은 `[시행 …]` 대괄호에서 뽑으므로 해당 없음.
    cover = "—"
    if doc.profile == "REGULATION":
        page1 = [e for e in doc.elements if e.page == 1]
        has_grid = any(e.type == "table" and e.attrs.get("grid") for e in page1)
        cover = "table+grid" if has_grid else "paragraph 흩어짐"

    missing = [k for k in _META_REQUIRED.get(doc.profile, ()) if not doc.meta.get(k)]
    meta_missing += [(name, k) for k in missing]
    meta_rows.append({
        "Document": name, "Profile": doc.profile, "표지 형태": cover,
        "effective_date": doc.meta.get("effective_date") or "—",
        "enacted_date": doc.meta.get("enacted_date") or "—",
        "revision": doc.meta.get("revision") or "—",
        "doc_no": doc.meta.get("doc_no") or "—",
        "⚠": "결측 " + ",".join(missing) if missing else "",
    })

META_DF = pd.DataFrame(meta_rows)
display(META_DF)

if meta_missing:
    print()
    print(f"⚠ 필수 문서 메타 결측 {len(meta_missing)}건 — 이대로 Chroma에 넣으면 "
          "시행일 필터가 조용히 어긋난다")
    for name, key in meta_missing:
        print(f"   - {name}: {key}")
else:
    print()
    print(f"✅ 필수 문서 메타 결측 0건 "
          f"(REGULATION·LAW {sum(1 for d in DOCS.values() if d.profile in _META_REQUIRED)}종)")

# 시행일이 서로 다른 값인지도 본다 — 전부 같으면 결손이 가려진다(위 사고의 원인)
_eff = {d.meta.get("effective_date") for d in DOCS.values()
        if d.profile == "REGULATION" and d.meta.get("effective_date")}
if len(_eff) == 1:
    print(f"※ 규정 시행일이 모두 {_eff.pop()}로 동일하다 — 값이 같으면 한둘이 비어도 "
          "눈에 안 띈다. 이 점검을 자동화로 남겨 두는 이유다.")

### 1-3. 청크 인벤토리

전 청크를 한 행씩 펼친다. 이후 모든 지표는 이 DataFrame과 원본 `Chunk` 객체에서 나온다.


In [3]:
inv_rows = []
for name, chunks in CHUNKS.items():
    doc = DOCS[name]
    types = {e.element_id: e.type for e in doc.elements}
    articles = {e.element_id: e.attrs.get("article_no") for e in doc.elements}
    for c in chunks:
        owned_arts = {articles.get(eid) for eid in c.element_ids} - {None}
        inv_rows.append({
            "Document": name,
            "Profile": doc.profile,
            "Chunk ID": c.chunk_id,
            "Type": c.chunk_type,
            "Role": c.chunk_role,
            "Size": c.size,
            "Citation": c.citation,
            "Chapter": c.chapter_title or "",
            "Article": c.article_label or "",
            "Clause": (f"{c.clause_start}~{c.clause_end}{c.clause_kind}"
                       if c.clause_start is not None else ""),
            "Pages": f"{c.page_start}-{c.page_end}",
            "Elements": len(c.element_ids),
            "Owned Articles": len(owned_arts),
            "Has Table": c.has_table,
            "Has List": c.has_list,
            "Flags": ",".join(c.flags),
            "Parent": c.parent_chunk_id or "",
            "Text": c.text[:200].replace("\n", " ⏎ "),
        })

chunk_df = pd.DataFrame(inv_rows)
print(f"인벤토리 {len(chunk_df):,}행")
display(chunk_df.head(12))


인벤토리 888행


,Document,Profile,Chunk ID,Type,Role,Size,Citation,Chapter,Article,Clause,Pages,Elements,Owned Articles,Has Table,Has List,Flags,Parent,Text
0,법인카드_사용규정,REGULATION,dump:법인카드_사용규정#s002#P,section,parent,387,법인카드_사용규정,,,,1-1,0,0,True,False,parent,,| 제정일 | 2026. 7. 20. | ⏎ |---|---| ⏎ | 시행일 | 2026. 8. 1. | ⏎ | 소관부...
1,법인카드_사용규정,REGULATION,dump:법인카드_사용규정#s002#01,table,child,72,법인카드_사용규정,,,,1-1,1,0,True,False,,dump:법인카드_사용규정#s002#P,| 제정일 | 2026. 7. 20. | ⏎ |---|---| ⏎ | 시행일 | 2026. 8. 1. | ⏎ | 소관부...
2,법인카드_사용규정,REGULATION,dump:법인카드_사용규정#s002#02,section,child,313,법인카드_사용규정,,,,1-1,3,0,False,False,,dump:법인카드_사용규정#s002#P,"개정이력 v1.0 제정 (조문 정합성 검토 반영) / v1.1 개정: 제2조 관리자 정의 정비, 제10조 사전승인 예외..."
3,법인카드_사용규정,REGULATION,dump:법인카드_사용규정#c01a001#01,article,atomic,152,법인카드_사용규정 제1조,제1장 총칙,제1조,,2-2,2,1,False,False,,,"이 규정은 타이거 주식회사(이하 ""회사"")가 임직원에게 발급하는 법인카드의 발급, 관리, 사용 및 정산에 관 한 사항을..."
4,법인카드_사용규정,REGULATION,dump:법인카드_사용규정#c01a002#01,article,atomic,560,법인카드_사용규정 제2조,제1장 총칙,제2조,,2-2,8,1,False,True,,,"이 규정에서 사용하는 용어의 정의는 다음과 같다. ⏎ ""법인카드""란 회사 명의로 발급되어 임직원이 업무 목적의 지출에 ..."
5,법인카드_사용규정,REGULATION,dump:법인카드_사용규정#c01a003#01,article,atomic,37,법인카드_사용규정 제3조,제1장 총칙,제3조,,2-2,2,1,False,False,,,이 규정은 회사로부터 법인카드를 발급받은 모든 임직원에게 적용한다.
6,법인카드_사용규정,REGULATION,dump:법인카드_사용규정#c02a004#01,article,atomic,145,법인카드_사용규정 제4조 제1~2항,제2장 법인카드의 발급 및 관리,제4조,1~2항,3-3,4,1,False,True,,,1. 팀장 이상 직책(팀장·부서장·본부장·대표이사)을 보임한 임직원에게는 원칙적으로 개인 법인카드를 발 급한다. ⏎ 2...
7,법인카드_사용규정,REGULATION,dump:법인카드_사용규정#c02a005#01,article,atomic,83,법인카드_사용규정 제5조,제2장 법인카드의 발급 및 관리,제5조,,3-3,2,1,False,False,,,"사용자는 소속 부서장의 승인을 받아 「법인카드 발급 신청서」를 경영지원본부에 제출하며, 경영지원본부는 신청 내용을 검토..."
8,법인카드_사용규정,REGULATION,dump:법인카드_사용규정#c02a006#01,article,atomic,224,법인카드_사용규정 제6조 제1~2항,제2장 법인카드의 발급 및 관리,제6조,1~2항,3-3,3,1,False,True,,,"1. 사용자는 법인카드를 선량한 관리자의 주의로 보관·사용하여야 하며, 타인에게 양도·대여할 수 없다. 다만 제4조제2..."
9,법인카드_사용규정,REGULATION,dump:법인카드_사용규정#c02a007#01,article,atomic,85,법인카드_사용규정 제7조,제2장 법인카드의 발급 및 관리,제7조,,3-3,3,1,False,False,,,"사용자는 법인카드의 분실 또는 도난을 인지한 즉시 카드사에 사용정지를 요청하고, 24시간 이내에 경영지 ⏎ 원본부에 서..."


---
## 2. ① 크기 분포 — Retrieval Suitability(크기 측면)

**크기는 목표가 아니라 제약이다.** 조 단위로 자른 결과가 임베딩에 맞는 범위에 들어왔는지만 본다.
짧은 청크가 곧 결함은 아니다 — 조 자체가 37자인 경우가 실제로 있다(`제3조 (적용범위)`).
**문제는 짧은 청크가 아니라 조각난 청크**이므로, 아래에서 `atomic`(조 통째)과 `child`(조각)를
나눠 센다.


In [4]:
def q(values, p: float):
    values = sorted(values)
    return values[min(len(values) - 1, int(round((len(values) - 1) * p)))] if values else 0


size_rows = []
for name, chunks in CHUNKS.items():
    leaves = [c for c in chunks if c.chunk_role != "parent"]
    sizes = [c.size for c in leaves]
    children = [c for c in leaves if c.chunk_role == "child"]
    size_rows.append({
        "Document": name,
        "Profile": DOCS[name].profile,
        "Leaves": len(leaves),
        "Min": min(sizes, default=0),
        "Median": int(st.median(sizes)) if sizes else 0,
        "p90": q(sizes, 0.90),
        "Max": max(sizes, default=0),
        f"> max({BUDGET.max})": sum(1 for s in sizes if s > BUDGET.max),
        f"> hard({BUDGET.hard})": sum(1 for s in sizes if s > BUDGET.hard),
        "Tiny(<100)": sum(1 for s in sizes if s < 100),
        "Fragment(<100 & child)": sum(1 for c in children if c.size < 100),
    })

size_df = pd.DataFrame(size_rows)
display(size_df)

all_sizes = [c.size for c in LEAVES]
frag = sum(1 for c in LEAVES if c.chunk_role == "child" and c.size < 100)
print(f"\n전체 잎 {len(all_sizes):,}개 · 중앙 {int(st.median(all_sizes))}자 · "
      f"p90 {q(all_sizes, .9)}자 · 최대 {max(all_sizes)}자")
print(f"max 초과 {sum(1 for s in all_sizes if s > BUDGET.max)}건 "
      f"/ hard 초과 {sum(1 for s in all_sizes if s > BUDGET.hard)}건")
print(f"100자 미만 {sum(1 for s in all_sizes if s < 100)}건 "
      f"— 그중 **조각(child)** 은 {frag}건 (나머지는 원래 짧은 조)")

print("\n크기 구간 분포")
BINS = [(0, 100), (100, 300), (300, 600), (600, 1000), (1000, 1200), (1200, 2000), (2000, 10**9)]
for lo, hi in BINS:
    n = sum(1 for s in all_sizes if lo <= s < hi)
    label = f"{lo:>5}~{hi if hi < 10**9 else '∞':>5}"
    print(f"  {label}  {n:>4}  {'█' * int(n / max(1, len(all_sizes)) * 60)}")


,Document,Profile,Leaves,Min,Median,p90,Max,> max(1200),> hard(2000),Tiny(<100),Fragment(<100 & child)
0,법인카드_사용규정,REGULATION,23,25,189,464,560,0,0,6,1
1,부서소개,DIAGRAM,6,6,435,624,802,0,0,1,0
2,업무추진비_사용규정,REGULATION,21,41,193,584,947,0,0,2,0
3,조직도,DIAGRAM,9,15,159,335,407,0,0,3,2
4,조직설계_상세기획서,DIAGRAM,26,18,155,395,1655,1,0,5,1
5,직급체계,DIAGRAM,7,6,249,362,539,0,0,3,2
6,출장비_사용규정,REGULATION,21,41,172,310,375,0,0,3,1
7,회식_운영규정,REGULATION,25,33,299,685,974,0,0,7,5
8,법인세법,LAW,377,5,360,981,1873,12,0,94,53
9,부가가치세법,LAW,115,23,441,994,1469,1,0,8,0



전체 잎 799개 · 중앙 290자 · p90 956자 · 최대 1873자
max 초과 14건 / hard 초과 0건
100자 미만 172건 — 그중 **조각(child)** 은 91건 (나머지는 원래 짧은 조)

크기 구간 분포
      0~  100   172  ████████████
    100~  300   239  █████████████████
    300~  600   174  █████████████
    600~ 1000   157  ███████████
   1000~ 1200    43  ███
   1200~ 2000    14  █
   2000~    ∞     0  


---
## 3. ② 요소 커버리지 — Semantic Integrity

**청킹에서 가장 치명적인 실패는 조용한 유실**이다. 파싱의 `test_no_content_loss`와 같은 계약을
여기서도 잰다.

- **소유(`element_ids`)**: 요소는 정확히 한 청크가 소유한다. 중복 소유는 검색 결과 중복이 된다.
- **참조(`context_element_ids`)**: 장 헤딩처럼 **헤더 문자열로만 실린 요소**와 표 청크에 복제된
  도입 문장. 유실이 아니므로 커버리지에는 넣되 소유와는 구분한다.


In [5]:
cov_rows = []
for name, chunks in CHUNKS.items():
    doc = DOCS[name]
    body = [e for e in doc.body() if e.text.strip() and not e.attrs.get("dropped")]
    body_ids = {e.element_id for e in body}
    owned_list = [eid for c in chunks for eid in c.element_ids]
    owned = set(owned_list)
    ctx = {eid for c in chunks for eid in c.context_element_ids}
    covered = owned | ctx
    cov_rows.append({
        "Document": name,
        "Body Elements": len(body_ids),
        "Owned": len(owned & body_ids),
        "Context-only": len((ctx - owned) & body_ids),
        "Uncovered": len(body_ids - covered),
        "Dup Ownership": len(owned_list) - len(owned),
        "Coverage": round(len(covered & body_ids) / len(body_ids), 4) if body_ids else 0.0,
    })

cov_df = pd.DataFrame(cov_rows)
display(cov_df)

UNCOVERED = []
for name, chunks in CHUNKS.items():
    doc = DOCS[name]
    covered = {eid for c in chunks for eid in c.element_ids}
    covered |= {eid for c in chunks for eid in c.context_element_ids}
    for e in doc.body():
        if e.text.strip() and not e.attrs.get("dropped") and e.element_id not in covered:
            UNCOVERED.append({"Document": name, "Element": e.element_id,
                              "Type": e.type, "Text": e.text[:80]})

total_body = cov_df["Body Elements"].sum()
COVERAGE = round((total_body - cov_df["Uncovered"].sum()) / total_body, 4)
DUP = int(cov_df["Dup Ownership"].sum())
print(f"\n전체 커버리지 {COVERAGE:.2%} · 미커버 {cov_df['Uncovered'].sum()}건 · 소유 중복 {DUP}건")
if UNCOVERED:
    print("\n⚠ 미커버 요소 (조용한 유실 — 반드시 원인 확인)")
    display(pd.DataFrame(UNCOVERED))
else:
    print("✅ 유실 0 — 모든 본문 요소가 청크에 담겼다")


,Document,Body Elements,Owned,Context-only,Uncovered,Dup Ownership,Coverage
0,법인카드_사용규정,102,94,8,0,0,1.0
1,부서소개,15,12,3,0,0,1.0
2,업무추진비_사용규정,86,80,6,0,0,1.0
3,조직도,25,23,2,0,0,1.0
4,조직설계_상세기획서,64,62,2,0,0,1.0
5,직급체계,20,19,1,0,0,1.0
6,출장비_사용규정,67,61,6,0,0,1.0
7,회식_운영규정,74,69,5,0,0,1.0
8,법인세법,1798,1794,4,0,0,1.0
9,부가가치세법,656,644,12,0,0,1.0



전체 커버리지 100.00% · 미커버 0건 · 소유 중복 0건
✅ 유실 0 — 모든 본문 요소가 청크에 담겼다


---
## 4. ③ 계층·인용 보존 — Context Completeness / Hierarchy Preservation

이 시스템의 요구는 "제9조 제3항 위반"으로 회계 담당자에게 제시하는 것이다(FR-RV).
따라서 **인용 문자열을 만들 수 없는 청크는 검색돼도 쓸 수 없다.**

`Article`이 비는 것이 전부 결함은 아니다 — 별표·부칙·서문과 도해형 문서는 애초에 조가 없다.
그래서 **조 체계가 있는 문서(REGULATION·LAW)의 본문 청크**로 한정해 다시 센다.


In [6]:
STRUCTURED = {"REGULATION", "LAW"}
NON_ARTICLE_TYPES = {"annex", "preamble", "section"}

hier_rows = []
for name, chunks in CHUNKS.items():
    leaves = [c for c in chunks if c.chunk_role != "parent"]
    body_leaves = [c for c in leaves
                   if DOCS[name].profile in STRUCTURED and c.chunk_type not in NON_ARTICLE_TYPES]
    def ratio(pred, pool):
        return round(sum(1 for c in pool if pred(c)) / len(pool), 4) if pool else None
    hier_rows.append({
        "Document": name,
        "Profile": DOCS[name].profile,
        "Leaves": len(leaves),
        "Citation": ratio(lambda c: bool(c.citation), leaves),
        "Header": ratio(lambda c: bool(c.header), leaves),
        "Article(all)": ratio(lambda c: bool(c.article_label), leaves),
        "Article(본문)": ratio(lambda c: bool(c.article_label), body_leaves),
        "Chapter": ratio(lambda c: bool(c.chapter_title), leaves),
        "Clause": ratio(lambda c: c.clause_start is not None, leaves),
    })

hier_df = pd.DataFrame(hier_rows)
display(hier_df)

structured_body = [c for name, chunks in CHUNKS.items() for c in chunks
                   if c.chunk_role != "parent" and DOCS[name].profile in STRUCTURED
                   and c.chunk_type not in NON_ARTICLE_TYPES]
CITATION_RATE = round(sum(1 for c in LEAVES if c.citation) / len(LEAVES), 4)
ARTICLE_RATE = round(sum(1 for c in structured_body if c.article_label) / len(structured_body), 4)
CHAPTER_RATE = round(sum(1 for c in LEAVES if c.chapter_title) / len(LEAVES), 4)
print(f"\n인용 문자열 보유 {CITATION_RATE:.2%} (전 청크)")
print(f"조 라벨 보유    {ARTICLE_RATE:.2%} (조 체계 문서의 본문 청크 {len(structured_body):,}개 기준)")
print(f"장 경로 보유    {CHAPTER_RATE:.2%}")

print("\n헤더 실물 — 청크 하나만 보고 소속을 알 수 있는가")
for c in LEAVES:
    if c.article_label and c.chapter_title and c.clause_start is not None:
        print(c.context_text()[:520]); break


,Document,Profile,Leaves,Citation,Header,Article(all),Article(본문),Chapter,Clause
0,법인카드_사용규정,REGULATION,23,1.0,1.0,0.8261,0.9500,0.8261,0.3913
1,부서소개,DIAGRAM,6,1.0,1.0,0.0000,NaN,0.0000,0.0000
2,업무추진비_사용규정,REGULATION,21,1.0,1.0,0.8095,1.0000,0.8095,0.3810
3,조직도,DIAGRAM,9,1.0,1.0,0.0000,NaN,0.0000,0.0000
4,조직설계_상세기획서,DIAGRAM,26,1.0,1.0,0.0000,NaN,0.0000,0.0000
5,직급체계,DIAGRAM,7,1.0,1.0,0.0000,NaN,0.0000,0.1429
6,출장비_사용규정,REGULATION,21,1.0,1.0,0.7143,0.9375,0.7143,0.3810
7,회식_운영규정,REGULATION,25,1.0,1.0,0.7600,0.9500,0.8400,0.1600
8,법인세법,LAW,377,1.0,1.0,0.9920,1.0000,0.9973,0.5172
9,부가가치세법,LAW,115,1.0,1.0,0.9826,1.0000,0.9913,0.7565



인용 문자열 보유 100.00% (전 청크)
조 라벨 보유    99.59% (조 체계 문서의 본문 청크 727개 기준)
장 경로 보유    91.36%

헤더 실물 — 청크 하나만 보고 소속을 알 수 있는가
[법인카드_사용규정 · v1.1 · 시행 2026.8.1]
제2장 법인카드의 발급 및 관리 > 제4조 (발급 대상)
▸ 법인카드_사용규정 제4조 제1~2항

1. 팀장 이상 직책(팀장·부서장·본부장·대표이사)을 보임한 임직원에게는 원칙적으로 개인 법인카드를 발 급한다.
2. 직책이 없는 임직원(비직책자)은 부서 공용카드를 사용하거나, 업무상 필요가 인정되는 경우 관리자 승
인을 받아 개인 카드를 발급받을 수 있다.


### 4-1. 문맥 자립도 대리지표

자동으로 잴 수 있는 것은 **형식**뿐이다. "청크 하나만 보고 이해되는가"의 최종 판단은 사람 몫이며
§10 검수 시트로 넘긴다. 여기서는 형식 실패 두 가지만 잡는다.

1. **지시어 시작** — `다만`·`이 경우`·`또한` 등으로 시작하는 청크는 앞 문맥 없이는 뜻이 서지 않는다.
   단, 우리 청커는 조 경계에서만 자르므로 **조 자체가 그렇게 시작하는 경우**와 구분해 센다.
2. **헤더 없음** — 계층 헤더가 비면 그 청크는 검색돼도 출처를 말할 수 없다.


In [7]:
DANGLING = re.compile(r"^\s*(다만|이 경우|또한|그러나|이때|이에|위 |해당 |상기 )")

dangling = [c for c in LEAVES if DANGLING.match(c.text)]
dangling_children = [c for c in dangling if c.chunk_role == "child"]
no_header = [c for c in LEAVES if not c.header]

print(f"지시어로 시작하는 청크 {len(dangling)}건 "
      f"({len(dangling) / len(LEAVES):.1%}) — 그중 **분할로 생긴 조각**은 {len(dangling_children)}건")
print(f"헤더 없는 청크 {len(no_header)}건")
if dangling:
    display(pd.DataFrame([{
        "Document": c.doc_name, "Chunk ID": c.chunk_id, "Role": c.chunk_role,
        "Citation": c.citation, "Text": c.text[:90].replace("\n", " ⏎ "),
    } for c in dangling[:15]]))

DANGLING_CHILD_RATE = round(len(dangling_children) / len(LEAVES), 4)


지시어로 시작하는 청크 0건 (0.0%) — 그중 **분할로 생긴 조각**은 0건
헤더 없는 청크 0건


---
## 5. ④ 경계 품질 — Boundary Quality

우리 청커는 **요소 경계를 넘지 않는 것**을 원칙으로 하므로, 문장 중간 절단은 분할 사다리의
마지막 단(문장·문자 분할)이 발동했을 때만 생긴다(`hard_split` 플래그).

**조 경계 순수성**은 별개다 — 한 청크가 서로 다른 조의 본문을 담으면 그 청크는 어느 조로도
인용할 수 없다. 소유 요소의 `article_no`(파싱이 붙인 값)로 직접 잰다.

두 가지를 **구분해서** 센다.

| 지표 | 뜻 | 판정 |
|---|---|---|
| **Cross-Article (raw)** | 소유 요소의 조 번호가 2개 이상 | 참고 — 파싱 태깅 잔재가 섞인다 |
| **Misattributed** | 그 조 번호가 **청크의 인용과 어긋남** | ❌ 거짓 인용. 0이어야 한다 |

> 📏 이 지표가 이 노트북의 최대 성과다. 초기 구현은 조 경계를 **64건** 넘고 있었다 —
> 법령에서 조의 상당수가 `heading`이 아니라 `paragraph`로 시작(`제17조(자본거래…) ① …`)하는데
> 헤딩만 경계로 삼았기 때문이다. 회귀 테스트로 고정했다
> (`test_article_starting_in_body_opens_its_own_block`).


In [8]:
bound_rows = []
CROSS = []
for name, chunks in CHUNKS.items():
    doc = DOCS[name]
    articles = {e.element_id: e.attrs.get("article_no") for e in doc.elements}
    leaves = [c for c in chunks if c.chunk_role != "parent"]
    cross = mis = 0
    for c in leaves:
        arts = {articles.get(eid) for eid in c.element_ids} - {None}
        if len(arts) <= 1:
            continue
        cross += 1
        # 청크가 인용한 조와 실제로 어긋나는가 = **거짓 인용**. 인용을 안 한 청크(서문 등)는
        # 어긋날 것이 없다.
        wrong = c.article_no is not None and (arts - {c.article_no})
        mis += bool(wrong)
        CROSS.append({"Document": name, "Chunk ID": c.chunk_id, "Citation": c.citation,
                      "Articles": sorted(arts), "Misattributed": bool(wrong),
                      "Text": c.text[:80].replace("\n", " ⏎ ")})
    bound_rows.append({
        "Document": name,
        "Leaves": len(leaves),
        "Cross-Article": cross,
        "Misattributed": mis,
        "Hard Split": sum(1 for c in leaves if "hard_split" in c.flags),
        "Clause Splits": REPORTS[name].splits.get("clause", 0),
        "List Splits": REPORTS[name].splits.get("list_item", 0),
        "Article Starts in Body": REPORTS[name].splits.get("article_start_in_body", 0),
    })

bound_df = pd.DataFrame(bound_rows)
display(bound_df)

CROSS_ARTICLE = int(bound_df["Cross-Article"].sum())
MISATTRIBUTED = int(bound_df["Misattributed"].sum())
HARD_SPLIT = int(bound_df["Hard Split"].sum())
print(f"\n조 경계 위반(raw) {CROSS_ARTICLE}건 · 그중 **거짓 인용** {MISATTRIBUTED}건 "
      f"· 문장 강제 분할 {HARD_SPLIT}건")
print("※ raw 위반에는 파싱(C2)이 연속 문장에 앞 조 번호를 붙여 둔 태깅 잔재가 섞인다.")
print("  실제로 문제인 것은 **청크의 인용과 어긋나는** 경우뿐이다.")
if CROSS:
    display(pd.DataFrame(CROSS))
else:
    print("✅ 한 청크가 두 조를 걸치는 경우 없음")

print("\n분할이 실제로 일어난 블록 — **이유 없는 분할이 있으면 안 된다**")
split_rows = []
for name, chunks in CHUNKS.items():
    for p in (c for c in chunks if c.chunk_role == "parent"):
        kids = [c for c in chunks if c.parent_chunk_id == p.chunk_id]
        reason = ("예산 초과" if p.size > BUDGET.max else
                  "표 분리" if any(k.has_table for k in kids) else "⚠ 이유 불명")
        split_rows.append({"Document": name, "Parent": p.chunk_id, "Citation": p.citation,
                           "Parent Size": p.size, "Children": len(kids), "Reason": reason})
split_df = pd.DataFrame(split_rows)
display(split_df["Reason"].value_counts().rename("blocks").to_frame())
display(split_df.head(10))


,Document,Leaves,Cross-Article,Misattributed,Hard Split,Clause Splits,List Splits,Article Starts in Body
0,법인카드_사용규정,23,0,0,0,0,0,0
1,부서소개,6,0,0,0,0,0,0
2,업무추진비_사용규정,21,0,0,0,1,0,0
3,조직도,9,0,0,0,0,0,0
4,조직설계_상세기획서,26,0,0,0,0,0,0
5,직급체계,7,0,0,0,0,0,0
6,출장비_사용규정,21,0,0,0,0,0,0
7,회식_운영규정,25,1,1,0,0,0,0
8,법인세법,377,1,0,0,59,83,85
9,부가가치세법,115,0,0,0,19,0,60



조 경계 위반(raw) 2건 · 그중 **거짓 인용** 1건 · 문장 강제 분할 0건
※ raw 위반에는 파싱(C2)이 연속 문장에 앞 조 번호를 붙여 둔 태깅 잔재가 섞인다.
  실제로 문제인 것은 **청크의 인용과 어긋나는** 경우뿐이다.


,Document,Chunk ID,Citation,Articles,Misattributed,Text
0,회식_운영규정,dump:회식_운영규정#c01a002#03,회식_운영규정 제2조,"[2, 6]",True,"구분 원칙: 참석자 전원이 임직원인 경우(①~⑦)는 복리후생비 또는 회의비로, 외부인(거래처·협력사 등)이 1인 이상 ..."
1,법인세법,dump:법인세법#s002#01,법인세법,"[55, 75]",False,제55조의2(토지등 양도소득에 대한 과세특례) ⏎ 제56조 제57조(외국 납부 세액공제 등) 제57조의2(간접투자회사 ...



분할이 실제로 일어난 블록 — **이유 없는 분할이 있으면 안 된다**


,blocks
Reason,
예산 초과,67
표 분리,22


,Document,Parent,Citation,Parent Size,Children,Reason
0,법인카드_사용규정,dump:법인카드_사용규정#s002#P,법인카드_사용규정,387,2,표 분리
1,법인카드_사용규정,dump:법인카드_사용규정#x001#P,법인카드_사용규정,697,2,표 분리
2,업무추진비_사용규정,dump:업무추진비_사용규정#c02a006#P,업무추진비_사용규정 제6조,1367,2,예산 초과
3,업무추진비_사용규정,dump:업무추진비_사용규정#x001#P,업무추진비_사용규정,445,2,표 분리
4,조직도,dump:조직도#s002#P,조직도,441,3,표 분리
5,조직설계_상세기획서,dump:조직설계_상세기획서#s003#P,조직설계_상세기획서,750,2,표 분리
6,조직설계_상세기획서,dump:조직설계_상세기획서#s021#P,조직설계_상세기획서,586,2,표 분리
7,조직설계_상세기획서,dump:조직설계_상세기획서#s022#P,조직설계_상세기획서,441,2,표 분리
8,조직설계_상세기획서,dump:조직설계_상세기획서#s023#P,조직설계_상세기획서,662,2,표 분리
9,직급체계,dump:직급체계#s004#P,직급체계,564,2,표 분리


---
## 6. ⑤ 표 무결성 — Table Integrity

표는 룰 임계값의 원천이다(`policies/tiger_tables.py`). 세 가지를 잰다.

1. **원자성** — 표 청크가 표 요소 하나만 소유하는가 (본문과 섞이지 않았는가)
2. **행 보존** — 원본 `grid`의 데이터 행이 청크 텍스트에 전부 남았는가
3. **헤더 반복** — 행 분할이 일어났다면 조각마다 헤더행이 있는가


In [9]:
def data_rows(el) -> list[list[str]]:
    grid = el.attrs.get("grid") or []
    if not grid or not isinstance(grid[0], list):
        return []
    return grid[int(el.attrs.get("header_rows") or 1):]


table_rows = []
for name, chunks in CHUNKS.items():
    doc = DOCS[name]
    by_id = {e.element_id: e for e in doc.elements}
    for el in doc.tables():
        holders = [c for c in chunks if el.element_id in c.element_ids and c.chunk_role != "parent"]
        rows = data_rows(el)
        joined = "\n".join(c.text for c in holders)
        kept = sum(1 for r in rows if all(str(cell).strip() in joined for cell in r if str(cell).strip()))
        mixed = any(len([e for e in c.element_ids if by_id.get(e) and by_id[e].type != "table"]) > 1
                    for c in holders)
        header_ok = all(
            all(str(h).strip() in c.text for h in (el.attrs.get("header_row") or [])[:2] if str(h).strip())
            for c in holders
        ) if holders else False
        table_rows.append({
            "Document": name,
            "Table": el.element_id,
            "Rows": len(rows),
            "Cols": el.attrs.get("cols", 0),
            "Chunks": len(holders),
            "Rows Kept": kept,
            "Row Retention": round(kept / len(rows), 3) if rows else None,
            "Header in All": header_ok,
            "Mixed w/ Text": mixed,
            "Annex": any(c.chunk_type == "annex" for c in holders),
        })

table_df = pd.DataFrame(table_rows)
display(table_df)

with_rows = table_df[table_df["Rows"] > 0]
ROW_RETENTION = round(with_rows["Row Retention"].mean(), 4) if len(with_rows) else None
TABLE_ATOMIC = round(1 - table_df["Mixed w/ Text"].mean(), 4) if len(table_df) else None
print(f"\n표 {len(table_df)}개 · 행 보존율 평균 {ROW_RETENTION:.2%} · 원자성 {TABLE_ATOMIC:.2%}")
print(f"본문과 섞인 표 {int(table_df['Mixed w/ Text'].sum())}건 · "
      f"두 청크 이상으로 쪼개진 표 {int((table_df['Chunks'] > 1).sum())}건")

print("\n핵심 한도표 실물 확인 (룰 임계값 원천)")
for c in ALL_CHUNKS:
    if "대표이사" in c.text and "300만원" in c.text and c.chunk_role != "parent":
        print(c.context_text()[:700]); break


,Document,Table,Rows,Cols,Chunks,Rows Kept,Row Retention,Header in All,Mixed w/ Text,Annex
0,법인카드_사용규정,법인카드_사용규정:p1:e3,2,2,1,2,1.0,True,False,False
1,법인카드_사용규정,법인카드_사용규정:p7:e112,5,4,1,5,1.0,True,False,True
2,부서소개,부서소개:p2:e7,7,5,1,7,1.0,True,False,False
3,부서소개,부서소개:p3:e14,5,5,1,5,1.0,True,False,False
4,부서소개,부서소개:p3:e18,3,5,1,3,1.0,True,False,False
5,부서소개,부서소개:p4:e24,1,5,1,1,1.0,True,False,False
6,부서소개,부서소개:p4:e26,11,3,1,11,1.0,True,False,False
7,업무추진비_사용규정,업무추진비_사용규정:p6:e96,3,3,1,3,1.0,True,False,True
8,업무추진비_사용규정,업무추진비_사용규정:p6:e99,4,2,1,4,1.0,True,False,True
9,조직도,조직도:p2:e5,8,2,1,8,1.0,True,False,False



표 39개 · 행 보존율 평균 100.00% · 원자성 100.00%
본문과 섞인 표 0건 · 두 청크 이상으로 쪼개진 표 0건

핵심 한도표 실물 확인 (룰 임계값 원천)
[법인카드_사용규정 · v1.1 · 시행 2026.8.1]
별표1. 직책별 법인카드 사용 한도 개정 v1.1
▸ 법인카드_사용규정

| 직책 | 1일 한도 | 월 한도 | 건당 사전승인 기준 |
|---|---|---|---|
| 대표이사 | 300만원 | 1,000만원 | 100만원 초과 |
| 본부장 | 250만원 | 800만원 | 80만원 초과 |
| 부서장 | 150만원 | 500만원 | 60만원 초과 |
| 팀장 | 100만원 | 400만원 | 50만원 초과 |
| 비직책자(공용카드) | 50만원 | 200만원 | 30만원 초과 |


---
## 7. ⑥ 노이즈 · 인용 신뢰도

**지우지 않고 표시만 한 것들**이다. 삭제 판단은 인덱싱 정책의 몫이고, 조용한 삭제는 금지다.

| 플래그 | 뜻 | 인덱싱 권고 |
|---|---|---|
| `toc_like` | 목차 잔재(조 제목 나열) | **제외 후보** |
| `marker_uncertain` | 파싱이 항 번호를 확신하지 못함 | 색인하되 인용 신뢰도 하향 |
| `table_row_split` | 표가 행으로 쪼개짐 | 그대로 |
| `hard_split` | 문장·문자 강제 분할 | 그대로(경계 품질 감점) |


In [10]:
flag_counter = Counter(f for c in ALL_CHUNKS for f in c.flags)
display(pd.DataFrame(sorted(flag_counter.items()), columns=["Flag", "Chunks"]))

noise = [c for c in LEAVES if {"toc_like", "marker_uncertain"} & set(c.flags)]
noise_df = pd.DataFrame([{
    "Document": c.doc_name, "Chunk ID": c.chunk_id, "Flags": ",".join(c.flags),
    "Size": c.size, "Citation": c.citation, "Text": c.text[:100].replace("\n", " ⏎ "),
} for c in noise])
if len(noise_df):
    display(noise_df)

print("\n파싱 단계에서 넘어온 리포트 (청킹이 소비한 계약)")
for name, doc in DOCS.items():
    skipped = doc.report.skipped_steps
    print(f"  {name:<20} skipped={list(skipped)} · dropped={len(doc.report.dropped_elements)} "
          f"· warnings={len(doc.report.warnings)}")

TOC_LIKE = flag_counter.get("toc_like", 0)
MARKER_UNCERTAIN = flag_counter.get("marker_uncertain", 0)


,Flag,Chunks
0,marker_uncertain,18
1,parent,89
2,toc_like,1


,Document,Chunk ID,Flags,Size,Citation,Text
0,법인세법,dump:법인세법#s002#01,toc_like,1125,법인세법,제55조의2(토지등 양도소득에 대한 과세특례) ⏎ 제56조 제57조(외국 납부 세액공제 등) 제57조의2(간접투자회사 ...
1,법인세법,dump:법인세법#c01a059#01,marker_uncertain,369,법인세법 제46조의3 제2항,제46조의3(적격분할 시 분할신설법인등에 대한 과세특례) ① 적격분할을 한 분할신설법인등은 제46조의2에도 불구하고 분...
2,법인세법,dump:법인세법#c01a059#02,marker_uncertain,1345,법인세법 제46조의3 제3~6항,③ 적격분할을 한 분할신설법인등은 3년 이내의 범위에서 대통령령으로 정하는 기간에 다음 각 호의 어느 하나에 해당하는 ...
3,법인세법,dump:법인세법#c01a060#01,marker_uncertain,846,법인세법 제46조의4 제2~3항,제46조의4(분할 시 이월결손금 등 공제 제한) ① 분할합병의 상대방법인의 분할등기일 현재 제13조제1항제1호의 결손금...
4,법인세법,dump:법인세법#c01a060#02,marker_uncertain,833,법인세법 제46조의4 제4~5항,④ 제46조의3제2항에 따라 분할신설법인등이 승계한 분할법인등의 감면 또는 세액공제는 분할법인등으로부터 승계받은 사업에...
5,법인세법,dump:법인세법#c01a060#03,marker_uncertain,408,법인세법 제46조의4 제7~8항,⑦ 분할법인등의 분할등기일 현재 기부금한도초과액으로서 제46조의3제2항에 따라 분할신설법인등이 승계한 금액은 분할신설법...
6,법인세법,dump:법인세법#c01a089#01,marker_uncertain,1078,법인세법 제62조의2 제2~4항,제62조의2(비영리내국법인의 자산양도소득에 대한 신고 특례) ① 비영리내국법인(제4조제3항제1호에 따른 수익사업을 하는...
7,법인세법,dump:법인세법#c01a089#02,marker_uncertain,674,법인세법 제62조의2 제5~9항,"⑤ 자산양도소득에 대한 과세표준의 계산에 관하여는 「소득세법」 제101조 및 제102조를 준용하고, 자산양도소득에 대한..."
8,법인세법,dump:법인세법#c04a167#02,marker_uncertain,827,법인세법 제98조 제2~7항,② 삭제<2022. 12. 31.> ⏎ ③ 삭제<2011. 12. 31.> ⏎ ④ 납세지 관할 세무서장은 원천징수의무자...
9,법인세법,dump:법인세법#c04a167#03,marker_uncertain,817,법인세법 제98조 제8~12항,"⑧ 외국법인에 건축, 건설, 기계장치 등의 설치ᆞ조립, 그 밖의 작업이나 그 작업의 지휘ᆞ감독 등에 관한 용역을 제공함..."



파싱 단계에서 넘어온 리포트 (청킹이 소비한 계약)
  법인카드_사용규정            skipped=['C4'] · dropped=0 · warnings=0
  부서소개                 skipped=['C4', 'C7'] · dropped=0 · warnings=0
  업무추진비_사용규정           skipped=['C4'] · dropped=0 · warnings=0
  조직도                  skipped=['C4', 'C7'] · dropped=0 · warnings=0
  조직설계_상세기획서           skipped=['C4', 'C7'] · dropped=0 · warnings=0
  직급체계                 skipped=['C4', 'C7'] · dropped=0 · warnings=0
  출장비_사용규정             skipped=['C4'] · dropped=0 · warnings=0
  회식_운영규정              skipped=['C4'] · dropped=0 · warnings=0
  법인세법                 skipped=['C5'] · dropped=5 · warnings=1
  부가가치세법               skipped=['C5'] · dropped=3 · warnings=1
  여신전문금융업법             skipped=['C5'] · dropped=3 · warnings=1


---
## 8. 전략 비교 (A~E)

같은 코퍼스에 다섯 전략을 걸어 **임베딩 없이 잴 수 있는 것만** 비교한다.

| | 전략 |
|---|---|
| **A** | Fixed-size 800자 |
| **B** | Fixed-size 800자 + overlap 100 |
| **C** | Recursive(문단→줄→문장→문자, max 800) |
| **D** | Structure-based (조 단위, 헤더 제외) |
| **E** | Hierarchy-aware + Parent-Child (**채택안**) |

> ⚠️ 이 표는 **구조 충실도**이지 검색 정확도가 아니다. Recall@K·MRR은 §11의 조건이 갖춰진 뒤에 잰다.


In [11]:
SEP = "\n"


def flatten(doc):
    """(문서 전체 텍스트, [(start, end, element)]) — span을 요소로 되돌리기 위한 색인."""
    spans, buf, pos = [], [], 0
    for el in doc.body():
        if el.attrs.get("dropped") or not el.text.strip():
            continue
        buf.append(el.text)
        spans.append((pos, pos + len(el.text), el))
        pos += len(el.text) + len(SEP)
    return SEP.join(buf), spans


def fixed_spans(text, size, overlap=0):
    out, i = [], 0
    step = size - overlap if size > overlap else size
    while i < len(text):
        out.append((i, min(i + size, len(text))))
        i += step
    return out


_SEPS = ["\n\n", "\n", "다. ", ". ", " "]


def recursive_spans(text, size, seps=_SEPS):
    """LangChain RecursiveCharacterTextSplitter와 같은 방식(구분자 우선순위 하강)."""
    out = []

    def rec(start, end, depth):
        if end - start <= size or depth >= len(seps):
            out.append((start, end))
            return
        pieces, cur = [], start
        for m in re.finditer(re.escape(seps[depth]), text[start:end]):
            cut = start + m.end()
            if cut - cur >= size:
                pieces.append((cur, cut))
                cur = cut
        pieces.append((cur, end))
        if len(pieces) == 1:
            rec(start, end, depth + 1)
            return
        for a, b in pieces:
            rec(a, b, depth + 1)

    rec(0, len(text), 0)
    return [(a, b) for a, b in out if b > a]


def score_spans(text, index, spans):
    sizes, cross, broken, mid, nocite = [], 0, 0, 0, 0
    for start, end in spans:
        sizes.append(end - start)
        els = [el for s, e, el in index if s < end and e > start]
        arts = {el.attrs.get("article_no") for el in els if el.attrs.get("article_no")}
        cross += len(arts) > 1
        nocite += len(arts) != 1
        for s, e, el in index:
            if el.type == "table" and s < end and e > start and (start > s or end < e):
                broken += 1
                break
        if end < len(text) and text[end - 1:end] not in ".\n。」 ":
            mid += 1
    return sizes, cross, broken, mid, nocite


LABELS = ["A_fixed800", "B_fixed800_ov100", "C_recursive800", "D_structure", "E_hier_parent"]
acc = {k: {"chunks": 0, "sizes": [], "cross": 0, "broken": 0, "mid": 0,
           "nocite": 0, "hier": 0} for k in LABELS}

for name, doc in DOCS.items():
    text, index = flatten(doc)
    for label, spans in (
        ("A_fixed800", fixed_spans(text, 800)),
        ("B_fixed800_ov100", fixed_spans(text, 800, 100)),
        ("C_recursive800", recursive_spans(text, 800)),
    ):
        sizes, cross, broken, mid, nocite = score_spans(text, index, spans)
        a = acc[label]
        a["chunks"] += len(spans); a["sizes"] += sizes
        a["cross"] += cross; a["broken"] += broken; a["mid"] += mid; a["nocite"] += nocite

    chunks = CHUNKS[name]
    leaves = [c for c in chunks if c.chunk_role != "parent"]
    articles = {e.element_id: e.attrs.get("article_no") for e in doc.elements}
    cross = sum(1 for c in leaves
                if c.article_no is not None
                and ({articles.get(e) for e in c.element_ids} - {None, c.article_no}))
    for label, count in (("D_structure", len(leaves)), ("E_hier_parent", len(chunks))):
        a = acc[label]
        a["chunks"] += count
        a["sizes"] += [c.size for c in leaves]
        a["cross"] += cross
        a["broken"] += sum(1 for c in leaves if "table_row_split" in c.flags)
        a["mid"] += sum(1 for c in leaves if "hard_split" in c.flags)
        a["nocite"] += sum(1 for c in leaves
                           if not c.article_label and doc.profile in STRUCTURED
                           and c.chunk_type not in NON_ARTICLE_TYPES)
        a["hier"] += sum(1 for c in leaves if c.header and (c.chapter_title or c.article_label)) \
            if label == "E_hier_parent" else 0

strategy_df = pd.DataFrame([{
    "Strategy": k,
    "Chunks": v["chunks"],
    "Median": int(st.median(v["sizes"])) if v["sizes"] else 0,
    "p90": q(v["sizes"], 0.9),
    "Max": max(v["sizes"], default=0),
    "Cross-Article": v["cross"],
    "Table Broken": v["broken"],
    "Mid-Sentence": v["mid"],
    "No Citation": v["nocite"],
    "Hierarchy Kept": v["hier"],
} for k, v in acc.items()])
display(strategy_df)


,Strategy,Chunks,Median,p90,Max,Cross-Article,Table Broken,Mid-Sentence,No Citation,Hierarchy Kept
0,A_fixed800,414,800,800,800,190,29,299,216,0
1,B_fixed800_ov100,471,800,800,800,218,30,333,246,0
2,C_recursive800,861,114,804,962,176,38,0,219,0
3,D_structure,799,290,956,1873,1,0,0,3,0
4,E_hier_parent,888,290,956,1873,1,0,0,3,730


---
## 9. 예산 민감도

`max`를 바꾸면 무엇이 얼마나 변하는지 본다. **크기 예산은 손잡이여야지 마법 상수가 아니다.**
임베딩 모델이 확정되면 이 표를 토큰 기준으로 다시 그려 최적값을 정한다(전략 §11 ①).


In [12]:
sweep_rows = []
for mx in SWEEP_MAXES:
    # hard도 함께 올린다 — max > hard면 분할 사다리의 마지막 칸이 죽어 hard가 도달
    # 불가가 된다(📏 이전 스윕의 max=2,400 행에서 2,392자 청크가 나온 경로). 이제
    # `Budget.__post_init__`이 그 조합을 막으므로 여기서 맞춰 준다.
    budget = Budget(target=int(mx * 0.67), max=mx, hard=max(Budget().hard, mx))
    chunks_n = leaves_n = parents = frag = cross = 0
    sizes = []
    for name, doc in DOCS.items():
        cs, _ = chunk_document(doc, budget)
        articles = {e.element_id: e.attrs.get("article_no") for e in doc.elements}
        lv = [c for c in cs if c.chunk_role != "parent"]
        chunks_n += len(cs); leaves_n += len(lv); parents += len(cs) - len(lv)
        sizes += [c.size for c in lv]
        frag += sum(1 for c in lv if c.chunk_role == "child" and c.size < 100)
        cross += sum(1 for c in lv if len({articles.get(e) for e in c.element_ids} - {None}) > 1)
    sweep_rows.append({
        "max": mx, "target": int(mx * 0.67), "Chunks": chunks_n, "Leaves": leaves_n,
        "Parents": parents, "Median": int(st.median(sizes)), "p90": q(sizes, .9),
        "Max": max(sizes), "Over max": sum(1 for s in sizes if s > mx),
        "Fragments(<100)": frag, "Cross-Article": cross,
    })

sweep_df = pd.DataFrame(sweep_rows)
display(sweep_df)
print("조 경계 위반은 예산과 무관하게 0이어야 한다 — 구조로 자르기 때문이다.")


,max,target,Chunks,Leaves,Parents,Median,p90,Max,Over max,Fragments(<100),Cross-Article
0,600,402,1106,952,154,318,613,1873,100,92,2
1,800,536,1009,880,129,318,708,1873,44,91,2
2,1200,804,888,799,89,290,956,1873,14,91,2
3,1600,1072,812,753,59,266,1121,1873,6,91,2
4,2400,1608,763,726,37,246,1183,2392,0,91,2


조 경계 위반은 예산과 무관하게 0이어야 한다 — 구조로 자르기 때문이다.


---
## 10. 사람 검수 시트 — 자동 판정이 불가능한 것

**Context Completeness의 최종 판단은 사람이 한다.** "이 청크만 보고 질문에 답할 수 있는가"는
자동으로 잴 수 없다. GT 없는 법령 3종을 사람 검수로 넘긴 파싱 평가와 같은 방식이다.

표본은 무작위가 아니라 **위험 구간 우선**으로 뽑는다 — 분할된 조각, 플래그 붙은 청크,
크기 양극단, 표 청크.


In [13]:
def review_bucket(c) -> str | None:
    if c.chunk_role == "parent":
        return None
    if {"toc_like", "marker_uncertain", "hard_split", "table_row_split"} & set(c.flags):
        return "flagged"
    if c.chunk_role == "child":
        return "split_child"
    if c.has_table:
        return "table"
    if c.size > BUDGET.max:
        return "oversize"
    if c.size < 100:
        return "tiny"
    return "normal"


buckets: dict[str, list] = {}
for c in ALL_CHUNKS:
    b = review_bucket(c)
    if b:
        buckets.setdefault(b, []).append(c)

PRIORITY = ["flagged", "oversize", "split_child", "table", "tiny", "normal"]
quota = {b: max(2, int(REVIEW_SAMPLE_N * w)) for b, w in
         zip(PRIORITY, [0.20, 0.15, 0.25, 0.15, 0.10, 0.15])}

samples = []
for b in PRIORITY:
    pool = buckets.get(b, [])
    picked = random.sample(pool, min(len(pool), quota[b]))
    for c in picked:
        samples.append({
            "Bucket": b,
            "Document": c.doc_name,
            "Chunk ID": c.chunk_id,
            "Type": c.chunk_type, "Role": c.chunk_role, "Size": c.size,
            "Citation": c.citation,
            "Flags": ",".join(c.flags),
            "Chunk (헤더 포함 전문)": c.context_text(),
            "Q1 이 청크만으로 뜻이 통하는가 (Y/N)": "",
            "Q2 인용(조·항)이 본문과 일치하는가 (Y/N)": "",
            "Q3 잘린 곳이 어색한가 (Y/N)": "",
            "메모": "",
        })

review_df = pd.DataFrame(samples)
print(f"검수 표본 {len(review_df)}건")
display(review_df["Bucket"].value_counts().rename("samples").to_frame())
display(review_df[["Bucket", "Document", "Citation", "Size", "Flags"]].head(15))


검수 표본 40건


,samples
Bucket,
split_child,10
flagged,8
oversize,6
table,6
normal,6
tiny,4


,Bucket,Document,Citation,Size,Flags
0,flagged,여신전문금융업법,여신전문금융업법 제16조 제2~6항,1014,marker_uncertain
1,flagged,법인세법,법인세법 제46조의3 제3~6항,1345,marker_uncertain
2,flagged,부가가치세법,부가가치세법 제8조 제2~5항,808,marker_uncertain
3,flagged,법인세법,법인세법 제98조 제8~12항,817,marker_uncertain
4,flagged,법인세법,법인세법 제46조의3 제2항,369,marker_uncertain
5,flagged,법인세법,법인세법 제98조 제2~7항,827,marker_uncertain
6,flagged,법인세법,법인세법 제46조의4 제7~8항,408,marker_uncertain
7,flagged,법인세법,법인세법 제62조의2 제2~4항,1078,marker_uncertain
8,oversize,법인세법,법인세법 제75조의8 제2~3항,1683,
9,oversize,법인세법,법인세법 제2조,1599,


---
## 11. 종합 점수

**정답 없는 항목은 점수화하지 않는다.** Retrieval Effectiveness는 `N/A`로 두고 가중치를
나머지 영역에 재분배한다(파싱 평가 노트북과 동일 규칙).


In [14]:
AREA_WEIGHTS = {"Structure Integrity": 0.40, "Context Sufficiency": 0.30,
                "Size Fitness": 0.30, "Retrieval Effectiveness": 0.00}

leaf_n = len(LEAVES)
within = sum(1 for c in LEAVES if c.size <= BUDGET.max) / leaf_n
frag_rate = sum(1 for c in LEAVES if c.chunk_role == "child" and c.size < 100) / leaf_n
over_hard = sum(1 for c in LEAVES if c.size > BUDGET.hard) / leaf_n

METRIC_SPEC = [
    ("Structure Integrity", "Element Coverage", COVERAGE, 0.35),
    ("Structure Integrity", "Ownership Uniqueness", 1.0 if DUP == 0 else 0.0, 0.15),
    ("Structure Integrity", "Citation Correctness", 1 - MISATTRIBUTED / leaf_n, 0.25),
    ("Structure Integrity", "Table Atomicity", TABLE_ATOMIC, 0.15),
    ("Structure Integrity", "Table Row Retention", ROW_RETENTION, 0.10),
    ("Context Sufficiency", "Citation Coverage", CITATION_RATE, 0.40),
    ("Context Sufficiency", "Article Label (본문)", ARTICLE_RATE, 0.30),
    ("Context Sufficiency", "Chapter Path", CHAPTER_RATE, 0.15),
    ("Context Sufficiency", "No Dangling Start", 1 - DANGLING_CHILD_RATE, 0.15),
    ("Size Fitness", "Within Budget", within, 0.45),
    ("Size Fitness", "No Hard Overflow", 1 - over_hard, 0.30),
    ("Size Fitness", "Fragment-Free", 1 - frag_rate, 0.25),
    ("Retrieval Effectiveness", "Recall@5", None, 0.50),
    ("Retrieval Effectiveness", "MRR", None, 0.50),
]

rows, area_scores = [], {}
for area in AREA_WEIGHTS:
    items = [(m, v, w) for a, m, v, w in METRIC_SPEC if a == area]
    scored = [(m, v, w) for m, v, w in items if v is not None]
    total_w = sum(w for _, _, w in scored)
    score = sum(v * w for _, v, w in scored) / total_w if total_w else None
    area_scores[area] = score
    for m, v, w in items:
        rows.append({"Category": area, "Metric": m,
                     "Score": None if v is None else round(v, 4),
                     "Weight": w, "Status": "N/A" if v is None else "scored"})
    rows.append({"Category": area, "Metric": "── 영역 점수",
                 "Score": None if score is None else round(score * 100, 1),
                 "Weight": AREA_WEIGHTS[area], "Status": "N/A" if score is None else "scored"})

live = {a: s for a, s in area_scores.items() if s is not None}
live_w = {a: AREA_WEIGHTS[a] for a in live}
norm = sum(live_w.values())
OVERALL = round(sum(live[a] * live_w[a] for a in live) / norm * 100, 1)

summary_df = pd.DataFrame(rows)
display(summary_df)

print(f"\n{'영역':<26}{'점수':>8}{'가중치':>8}")
for area, score in area_scores.items():
    w = AREA_WEIGHTS[area] / norm if score is not None else 0
    shown = f"{score * 100:.1f}" if score is not None else "N/A"
    print(f"{area:<26}{shown:>8}{w:>8.2f}")
print(f"\n★ Overall (N/A 제외 재분배)  {OVERALL}/100")
print("  ※ 검색 성능은 미포함이다. 이 점수는 '구조가 온전한가'까지만 말한다.")


,Category,Metric,Score,Weight,Status
0,Structure Integrity,Element Coverage,1.0000,0.35,scored
1,Structure Integrity,Ownership Uniqueness,1.0000,0.15,scored
2,Structure Integrity,Citation Correctness,0.9987,0.25,scored
3,Structure Integrity,Table Atomicity,1.0000,0.15,scored
4,Structure Integrity,Table Row Retention,1.0000,0.10,scored
5,Structure Integrity,── 영역 점수,100.0000,0.40,scored
6,Context Sufficiency,Citation Coverage,1.0000,0.40,scored
7,Context Sufficiency,Article Label (본문),0.9959,0.30,scored
8,Context Sufficiency,Chapter Path,0.9136,0.15,scored
9,Context Sufficiency,No Dangling Start,1.0000,0.15,scored



영역                              점수     가중치
Structure Integrity          100.0    0.40
Context Sufficiency           98.6    0.30
Size Fitness                  96.4    0.30
Retrieval Effectiveness        N/A    0.00

★ Overall (N/A 제외 재분배)  98.5/100
  ※ 검색 성능은 미포함이다. 이 점수는 '구조가 온전한가'까지만 말한다.


### 11-1. 차트


In [15]:
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib import font_manager

    for cand in ("Malgun Gothic", "NanumGothic", "AppleGothic"):
        if any(cand == f.name for f in font_manager.fontManager.ttflist):
            plt.rcParams["font.family"] = cand
            break
    plt.rcParams["axes.unicode_minus"] = False

    fig, axes = plt.subplots(2, 2, figsize=(13, 8))

    axes[0][0].hist(all_sizes, bins=40, color="#4C78A8")
    axes[0][0].axvline(BUDGET.max, color="#E45756", ls="--", label=f"max {BUDGET.max}")
    axes[0][0].axvline(BUDGET.target, color="#F58518", ls=":", label=f"target {BUDGET.target}")
    axes[0][0].set_title("chunk size distribution"); axes[0][0].legend()

    areas = [a for a, s in area_scores.items() if s is not None]
    axes[0][1].barh(areas, [area_scores[a] * 100 for a in areas], color="#54A24B")
    axes[0][1].set_xlim(0, 100); axes[0][1].set_title(f"area scores (overall {OVERALL})")

    axes[1][0].bar(strategy_df["Strategy"], strategy_df["Cross-Article"], color="#E45756")
    axes[1][0].set_title("cross-article violations by strategy")
    axes[1][0].tick_params(axis="x", rotation=30, labelsize=8)

    axes[1][1].plot(sweep_df["max"], sweep_df["Leaves"], marker="o", label="leaves")
    axes[1][1].plot(sweep_df["max"], sweep_df["Fragments(<100)"], marker="s", label="fragments")
    axes[1][1].set_title("budget sweep"); axes[1][1].set_xlabel("max chars"); axes[1][1].legend()

    fig.tight_layout()
    fig.savefig(CHUNK_DIR / "chunking_charts.png", dpi=140)
    print(f"차트 저장: {CHUNK_DIR / 'chunking_charts.png'}")
    plt.close(fig)
except ImportError:
    print("matplotlib 없음 — 차트를 건너뛴다(지표 CSV는 그대로 생성된다)")


차트 저장: D:\project\SKN29-FINAL-1TEAM\docling_eval\output\chunking\chunking_charts.png


---
## 12. 산출물 저장 (`output/chunking/`)

| 파일 | 내용 |
|---|---|
| `chunk_inventory.csv` | 전 청크 1행씩 — 무엇이 만들어졌는지의 원장 |
| `chunk_details.csv` | 문서별 지표(크기·커버리지·계층·경계) |
| `chunk_summary.csv` | Category / Metric / Score 요약 |
| `strategy_comparison.csv` | 전략 A~E 비교 |
| `budget_sweep.csv` | 예산 민감도 |
| `table_integrity.csv` | 표별 행 보존·원자성 |
| `review_samples.csv` | **사람 검수 시트**(Q1~Q3 빈칸) |
| `uncovered_elements.csv` | 미커버 요소(있을 때만) |
| `chunking_report.md` | 종합 리포트 |
| `chunking_charts.png` | 차트 |


In [16]:
detail_df = (size_df
             .merge(cov_df, on="Document", how="left")
             .merge(hier_df.drop(columns=["Profile", "Leaves"]), on="Document", how="left")
             .merge(bound_df.drop(columns=["Leaves"]), on="Document", how="left"))

outputs = {
    "chunk_inventory.csv": chunk_df,
    "chunk_details.csv": detail_df,
    "chunk_summary.csv": summary_df,
    "strategy_comparison.csv": strategy_df,
    "budget_sweep.csv": sweep_df,
    "table_integrity.csv": table_df,
    "review_samples.csv": review_df,
}
if UNCOVERED:
    outputs["uncovered_elements.csv"] = pd.DataFrame(UNCOVERED)
if len(noise_df):
    outputs["flagged_chunks.csv"] = noise_df

for filename, frame in outputs.items():
    frame.to_csv(CHUNK_DIR / filename, index=False, encoding="utf-8-sig")
    print(f"  {filename:<28} {len(frame):>5}행")


  chunk_inventory.csv            888행
  chunk_details.csv               11행
  chunk_summary.csv               18행
  strategy_comparison.csv          5행
  budget_sweep.csv                 5행
  table_integrity.csv             39행
  review_samples.csv              40행
  flagged_chunks.csv              19행


In [17]:
def df_to_md(df: pd.DataFrame, max_rows: int = 40) -> str:
    view = df.head(max_rows)
    head = "| " + " | ".join(str(c) for c in view.columns) + " |"
    sep = "|" + "---|" * len(view.columns)
    body = ["| " + " | ".join("" if pd.isna(v) else str(v) for v in row) + " |"
            for row in view.itertuples(index=False)]
    tail = f"\n\n_({len(df) - max_rows}행 생략 — CSV 참조)_" if len(df) > max_rows else ""
    return "\n".join([head, sep, *body]) + tail


def verdict(value, hi=0.95, lo=0.85) -> str:
    if value is None:
        return "N/A"
    return "✅" if value >= hi else ("⚠️" if value >= lo else "❌")


lines = [
    "# 청킹 평가 리포트",
    "",
    f"- 대상: **{len(DOCS)}종** / 요소 {sum(len(d.body()) for d in DOCS.values()):,}개",
    f"- 산출: **청크 {len(ALL_CHUNKS):,}개** (잎 {len(LEAVES):,} / 부모 {len(ALL_CHUNKS)-len(LEAVES)})",
    f"- 예산: target {BUDGET.target} · max {BUDGET.max} · hard {BUDGET.hard} (문자)",
    f"- 전략 캐논: `llm_wiki/_context/chunking-strategy.md`",
    "",
    f"## 종합 {OVERALL}/100",
    "",
    "> 검색 성능(Recall@K·MRR)은 임베딩 모델·질의 정답셋 부재로 **N/A**이며 가중치를 재분배했다.",
    "> 이 점수는 '구조가 온전한가'까지만 말한다.",
    "",
    "| 영역 | 점수 | 가중치(재분배) |",
    "|---|---|---|",
]
for area, score in area_scores.items():
    w = AREA_WEIGHTS[area] / norm if score is not None else 0
    lines.append(f"| {area} | {'N/A' if score is None else f'{score*100:.1f}'} | {w:.2f} |")

lines += [
    "",
    "## 핵심 지표",
    "",
    "| 지표 | 값 | 판정 |",
    "|---|---|---|",
    f"| 요소 커버리지 (유실 0인가) | {COVERAGE:.2%} | {verdict(COVERAGE, 1.0, 0.99)} |",
    f"| 소유 중복 | {DUP}건 | {'✅' if DUP == 0 else '❌'} |",
    f"| 거짓 인용(오귀속) | {MISATTRIBUTED}건 | {'✅' if MISATTRIBUTED == 0 else '❌'} |",
    f"| 조 경계 위반(raw, 태깅 잔재 포함) | {CROSS_ARTICLE}건 | {'✅' if CROSS_ARTICLE <= 2 else '⚠️'} |",
    f"| 인용 문자열 보유 | {CITATION_RATE:.2%} | {verdict(CITATION_RATE)} |",
    f"| 조 라벨 보유(본문) | {ARTICLE_RATE:.2%} | {verdict(ARTICLE_RATE)} |",
    f"| 표 원자성 | {TABLE_ATOMIC:.2%} | {verdict(TABLE_ATOMIC)} |",
    f"| 표 행 보존율 | {ROW_RETENTION:.2%} | {verdict(ROW_RETENTION)} |",
    f"| 문장 강제 분할 | {HARD_SPLIT}건 | {'✅' if HARD_SPLIT == 0 else '⚠️'} |",
    f"| 예산 초과(max) | {sum(1 for s in all_sizes if s > BUDGET.max)}건 | — |",
    f"| 노이즈 표시 | toc_like {TOC_LIKE} · marker_uncertain {MARKER_UNCERTAIN} | 표시만, 삭제 없음 |",
    "",
]
mis_rows = [r for r in CROSS if r["Misattributed"]]
if mis_rows:
    lines += ["> **거짓 인용 상세** — 아래는 청크의 인용과 어긋난 청크다. "
              "원인이 청킹인지 파싱 태깅(C2)인지 매번 확인할 것.", ""]
    lines += [f"> - `{r['Chunk ID']}` — 인용 `{r['Citation']}` vs 요소 조번호 {r['Articles']}"
              for r in mis_rows]
    lines += [""]
lines += [
    "## 크기 분포",
    "",
    f"중앙 {int(st.median(all_sizes))}자 · p90 {q(all_sizes, .9)}자 · 최대 {max(all_sizes)}자",
    "",
    df_to_md(size_df),
    "",
    "## 전략 비교 (구조 충실도)",
    "",
    df_to_md(strategy_df),
    "",
    "## 예산 민감도",
    "",
    df_to_md(sweep_df),
    "",
    "## 표 무결성",
    "",
    df_to_md(table_df, 30),
    "",
    "## 남은 일",
    "",
    "1. **임베딩 모델 확정** → 예산을 토큰 기준으로 재보정, Chroma upsert 착수",
    "2. **질의 정답셋 30건**(정답은 청크 ID가 아니라 **조문 ID**로) → Recall@5·MRR 측정",
    f"3. **사람 검수** — `review_samples.csv` {len(review_df)}건의 Q1~Q3 채우기",
    "",
    "_생성: `docling_eval/chunking_evaluation.ipynb`_",
]

report_path = CHUNK_DIR / "chunking_report.md"
report_path.write_text("\n".join(lines), encoding="utf-8")
print(f"리포트 저장: {report_path}")
print()
print("\n".join(lines[:40]))


리포트 저장: D:\project\SKN29-FINAL-1TEAM\docling_eval\output\chunking\chunking_report.md

# 청킹 평가 리포트

- 대상: **11종** / 요소 3,717개
- 산출: **청크 888개** (잎 799 / 부모 89)
- 예산: target 800 · max 1200 · hard 2000 (문자)
- 전략 캐논: `llm_wiki/_context/chunking-strategy.md`

## 종합 98.5/100

> 검색 성능(Recall@K·MRR)은 임베딩 모델·질의 정답셋 부재로 **N/A**이며 가중치를 재분배했다.
> 이 점수는 '구조가 온전한가'까지만 말한다.

| 영역 | 점수 | 가중치(재분배) |
|---|---|---|
| Structure Integrity | 100.0 | 0.40 |
| Context Sufficiency | 98.6 | 0.30 |
| Size Fitness | 96.4 | 0.30 |
| Retrieval Effectiveness | N/A | 0.00 |

## 핵심 지표

| 지표 | 값 | 판정 |
|---|---|---|
| 요소 커버리지 (유실 0인가) | 100.00% | ✅ |
| 소유 중복 | 0건 | ✅ |
| 거짓 인용(오귀속) | 1건 | ❌ |
| 조 경계 위반(raw, 태깅 잔재 포함) | 2건 | ✅ |
| 인용 문자열 보유 | 100.00% | ✅ |
| 조 라벨 보유(본문) | 99.59% | ✅ |
| 표 원자성 | 100.00% | ✅ |
| 표 행 보존율 | 100.00% | ✅ |
| 문장 강제 분할 | 0건 | ✅ |
| 예산 초과(max) | 14건 | — |
| 노이즈 표시 | toc_like 1 · marker_uncertain 18 | 표시만, 삭제 없음 |

> **거짓 인용 상세** — 아래는 청크의 인용과 어긋난 청크다. 원인이 청킹인지 파싱 태깅(C2)인지 매번 확인할 것.

> - `dump:회식_

---
## 13. 이 평가가 말하지 않는 것

1. **검색이 잘 되는지는 모른다.** 구조가 온전하다는 것과 질의에 맞는 청크가 상위에 뜬다는 것은
   다른 문제다. §11의 Retrieval Effectiveness가 `N/A`인 이유이고, 그 칸을 채우려면
   임베딩 모델과 질의 정답셋이 먼저 필요하다.
2. **청크가 "이해되는가"는 사람만 안다.** §10 시트의 Q1~Q3가 채워지기 전까지 Context
   Sufficiency는 형식 지표(헤더·인용 보유율)에 불과하다.
3. **파싱 품질은 여기서 재지 않는다.** 청킹은 파싱이 준 계층을 신뢰하고 자를 뿐이다.
   계층이 틀렸다면 `docling_parsing_evaluation.ipynb`가 잡을 문제다.
